### Target Synthensis

In [ ]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # Months: July 2025 to December 2050
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]

DEFAULT_NOISE = 0.005  # Very small noise

SHRINK_TOWARD_MEAN = 0.5  # How much to shrink predictions toward historical mean
APPLY_SMOOTHING = True
SMOOTHING_WINDOW = 3

# === LOAD DATA === #
df = pd.read_csv(FILENAME)
df['date'] = pd.to_datetime(df['date'])

# Save original column order, excluding targets
original_order = [col for col in df.columns if col not in TARGET_COLUMNS]

# Determine dynamic features
excluded_cols = set(STATIC_FEATURES + TARGET_COLUMNS + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()
    
    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        
        if ts['y'].nunique() < 2:
            continue  # Skip features with no variance

        model = Prophet(
            yearly_seasonality=True,
            daily_seasonality=False,
            weekly_seasonality=False,
            seasonality_mode='additive'
        )

        # Comment out biannual if needed
        # model.add_seasonality(name='biannual', period=182.5, fourier_order=3)

        try:
            model.fit(ts)
        except Exception as e:
            print(f"⚠️ Failed to fit Prophet for {feature} in {city}: {e}")
            continue

        future = model.make_future_dataframe(periods=FUTURE_PERIODS, freq='MS')
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS).rename(columns={'ds': 'date', 'yhat': feature})

        # Minimal noise
        std_dev = ts['y'].std()
        noise = np.random.normal(0, DEFAULT_NOISE * std_dev, size=len(predicted))
        predicted[feature] += noise

        # Shrink toward historical mean
        historical_mean = ts['y'].mean()
        predicted[feature] = (
            SHRINK_TOWARD_MEAN * historical_mean +
            (1 - SHRINK_TOWARD_MEAN) * predicted[feature]
        )

        # Hard clipping: stay within 5th–90th percentile
        lower_bound = ts['y'].quantile(0.05)
        upper_bound = ts['y'].quantile(0.90)
        predicted[feature] = predicted[feature].clip(lower=lower_bound, upper=upper_bound)

        # Optional: Smooth results
        if APPLY_SMOOTHING:
            predicted[feature] = predicted[feature].rolling(window=SMOOTHING_WINDOW, min_periods=1).mean()

        # Merge
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column explicitly if missing
    if 'city' not in forecasted_features.columns:
        forecasted_features['city'] = city

    # Reorder columns to match original file (excluding targets)
    final_columns = [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[['date'] + final_columns]

    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

# Save output
final_synthetic_df.to_csv("datasets/synthetic_features_small_conservative.csv", index=False)
print("✅ Conservative synthetic dataset saved as 'synthetic_features_small_conservative.csv'")


### Full Synthesis

In [ ]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # Months: July 2025 to December 2050
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]

DEFAULT_NOISE = 0.01  # Small noise
SHRINK_TOWARD_MEAN = 0.3  # Gently shrink to historical mean
APPLY_SMOOTHING = True
SMOOTHING_WINDOW = 3

# === LOAD DATA === #
df = pd.read_csv(FILENAME)
df['date'] = pd.to_datetime(df['date'])

# Save original column order
original_order = df.columns.tolist()

# Dynamic features now includes both predictors + target columns
excluded_cols = set(STATIC_FEATURES + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()
    
    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        
        if ts['y'].nunique() < 2:
            continue  # Skip constant features

        model = Prophet(
            yearly_seasonality=True,
            daily_seasonality=False,
            weekly_seasonality=False,
            seasonality_mode='additive'
        )

        try:
            model.fit(ts)
        except Exception as e:
            print(f"⚠️ Failed to fit Prophet for {feature} in {city}: {e}")
            continue

        future = model.make_future_dataframe(periods=FUTURE_PERIODS, freq='MS')
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS).rename(columns={'ds': 'date', 'yhat': feature})

        # Add small Gaussian noise
        std_dev = ts['y'].std()
        noise = np.random.normal(0, DEFAULT_NOISE * std_dev, size=len(predicted))
        predicted[feature] += noise

        # Shrink forecast gently toward historical mean
        hist_mean = ts['y'].mean()
        predicted[feature] = (
            SHRINK_TOWARD_MEAN * hist_mean +
            (1 - SHRINK_TOWARD_MEAN) * predicted[feature]
        )

        # Clip to realistic bounds (5th–90th percentile)
        q5 = ts['y'].quantile(0.05)
        q90 = ts['y'].quantile(0.90)
        predicted[feature] = predicted[feature].clip(lower=q5, upper=q90)

        # Optional smoothing
        if APPLY_SMOOTHING:
            predicted[feature] = predicted[feature].rolling(window=SMOOTHING_WINDOW, min_periods=1).mean()

        # Merge feature
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column explicitly if missing
    if 'city' not in forecasted_features.columns:
        forecasted_features['city'] = city

    # Reorder to match original file (now includes targets)
    reordered = ['date'] + [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[[col for col in reordered if col in forecasted_features.columns]]

    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

# Save output
final_synthetic_df.to_csv("datasets/synthetic_all_features_incl_targets.csv", index=False)
print("✅ Full synthetic dataset with targets saved as 'synthetic_all_features_incl_targets.csv'")


In [ ]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np
from sklearn.utils import resample

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # Months: July 2025 to December 2050
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]
DEFAULT_NOISE = 0.01  # Small noise for synthetic data
SHRINK_TOWARD_MEAN = 0.3  # Shrink predictions toward historical mean
APPLY_SMOOTHING = True
SMOOTHING_WINDOW = 3
DOWNSAMPLE_FREQ = '3M'  # Downsample to quarterly data
MAX_SAMPLES_PER_CITY = 100  # Cap samples per city for balancing

# === LOAD DATA === #
try:
    df = pd.read_csv(FILENAME)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
except Exception as e:
    print(f"❌ Error loading CSV: {e}")
    raise

# Ensure all expected columns exist
missing_cols = [col for col in STATIC_FEATURES + TARGET_COLUMNS + ['date'] if col not in df.columns]
if missing_cols:
    print(f"❌ Missing columns in dataset: {missing_cols}")
    raise ValueError("Dataset missing required columns")

# Save original column order
original_order = df.columns.tolist()

# Dynamic features include predictors + target columns
excluded_cols = set(STATIC_FEATURES + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# === DIAGNOSTICS FOR KAMPALA === #
kampala_df = df[df['city'] == 'Kampala']
if not kampala_df.empty:
    print("Kampala Data Summary:")
    print(kampala_df[TARGET_COLUMNS].describe())
    for col in TARGET_COLUMNS:
        if col in kampala_df.columns:
            print(f"Kampala {col} unique values: {kampala_df[col].nunique()}")
            q1, q3 = kampala_df[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            outliers = ((kampala_df[col] < q1 - 1.5 * iqr) | (kampala_df[col] > q3 + 1.5 * iqr)).sum()
            print(f"Kampala {col} outliers (beyond 1.5*IQR): {outliers}")
else:
    print("⚠️ No data for Kampala found")

# === DOWNSAMPLING AND BALANCING FUNCTION === #
def downsample_and_balance(city_df, freq=DOWNSAMPLE_FREQ, max_samples=MAX_SAMPLES_PER_CITY):
    try:
        # Ensure date is index for resampling
        city_df = city_df.set_index('date')
        # Resample to specified frequency, aggregating numeric and static columns
        agg_dict = {col: 'mean' for col in dynamic_features if col in city_df.columns}
        agg_dict.update({col: 'first' for col in STATIC_FEATURES if col in city_df.columns})
        city_df = city_df.resample(freq).agg(agg_dict).reset_index()

        # Handle missing values after resampling
        city_df = city_df.fillna(method='ffill').fillna(method='bfill')

        # Cap samples to balance dataset
        if len(city_df) > max_samples:
            city_df = resample(city_df, n_samples=max_samples, random_state=42)

        # Stratified sampling for target columns
        for target in TARGET_COLUMNS:
            if target in city_df.columns and city_df[target].nunique() > 1:
                try:
                    bins = pd.qcut(city_df[target], q=4, duplicates='drop', labels=False)
                    city_df = city_df.groupby(bins, group_keys=False).apply(
                        lambda x: resample(x, n_samples=min(len(x), max_samples // 4), random_state=42)
                    ).reset_index(drop=True)
                except Exception as e:
                    print(f"⚠️ Failed to stratify {target} for city: {e}")
                    continue

        return city_df
    except Exception as e:
        print(f"❌ Error in downsample_and_balance: {e}")
        return city_df

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    
    # Downsample and balance historical data
    city_df = downsample_and_balance(city_df)
    if city_df.empty:
        print(f"⚠️ No data after downsampling for {city}")
        continue
    
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()
    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        if feature not in city_df.columns:
            continue
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        
        if ts['y'].nunique() < 2 or ts['y'].isna().all():
            print(f"⚠️ Skipping {feature} for {city}: insufficient unique values or all NaN")
            continue

        model = Prophet(
            yearly_seasonality=True,
            daily_seasonality=False,
            weekly_seasonality=False,
            seasonality_mode='additive'
        )

        try:
            model.fit(ts)
        except Exception as e:
            print(f"⚠️ Failed to fit Prophet for {feature} in {city}: {e}")
            continue

        # Create future dates with downsampled frequency
        future = model.make_future_dataframe(periods=FUTURE_PERIODS // 3, freq=DOWNSAMPLE_FREQ)
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS // 3).rename(columns={'ds': 'date', 'yhat': feature})

        # Add small Gaussian noise
        std_dev = ts['y'].std()
        if not np.isnan(std_dev):
            noise = np.random.normal(0, DEFAULT_NOISE * std_dev, size=len(predicted))
            predicted[feature] += noise

        # Shrink forecast toward historical mean
        hist_mean = ts['y'].mean()
        if not np.isnan(hist_mean):
            predicted[feature] = (
                SHRINK_TOWARD_MEAN * hist_mean +
                (1 - SHRINK_TOWARD_MEAN) * predicted[feature]
            )

        # Clip to realistic bounds (5th–90th percentile)
        q5 = ts['y'].quantile(0.05)
        q90 = ts['y'].quantile(0.90)
        if not (np.isnan(q5) or np.isnan(q90)):
            predicted[feature] = predicted[feature].clip(lower=q5, upper=q90)

        # Optional smoothing
        if APPLY_SMOOTHING:
            predicted[feature] = predicted[feature].rolling(window=SMOOTHING_WINDOW, min_periods=1).mean()

        # Merge feature
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date', how='outer')

    # Handle missing values after merging
    forecasted_features = forecasted_features.fillna(method='ffill').fillna(method='bfill')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column explicitly
    forecasted_features['city'] = city

    # Reorder to match original file
    reordered = ['date'] + [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[[col for col in reordered if col in forecasted_features.columns]]

    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
if synthetic_data_all:
    final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

    # Balance final dataset across cities
    city_counts = final_synthetic_df['city'].value_counts()
    min_samples = city_counts.min()
    balanced_df = pd.concat([
        resample(final_synthetic_df[final_synthetic_df['city'] == city], n_samples=min_samples, random_state=42)
        for city in final_synthetic_df['city'].unique()
    ], ignore_index=True)

    # Save output
    balanced_df.to_csv("datasets/synthetic_downsampled_balanced.csv", index=False)
    print("✅ Downsampled and balanced synthetic dataset saved as 'synthetic_downsampled_balanced.csv'")
else:
    print("❌ No synthetic data generated")

# Final diagnostics
if 'balanced_df' in locals():
    print("Final Dataset Summary:")
    print(balanced_df['city'].value_counts())
    print(balanced_df[TARGET_COLUMNS].describe())

### Sarima Forecasting.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
csv_file = 'datasets/city_timeseries_synthetic_dataset.csv'  # Replace with your file path
try:
    df = pd.read_csv(csv_file, skipinitialspace=True, encoding='utf-8')
except Exception as e:
    print(f"Error loading CSV: {e}")
    raise

# Verify columns
required_columns = ['date', 'city', 'monsoon_intensity', 'climate_change', 'siltation',
                   'landslide_risks', 'rainfall_mm', 'temperature_2m', 'runoff', 'total_precipitation',
                   'high_vegetation_cover', 'geopotential_height', 'soil_volume_water_content_level1',
                   'soil_volume_water_content_level2', 'soil_volume_water_content_level3',
                   'soil_volume_water_content_level4']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}. Available columns: {df.columns.tolist()}")

# Convert date to datetime (YYYY-MM-DD)
try:
    df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce')
except Exception as e:
    print(f"Error converting 'date' to datetime: {e}")
    raise

# Handle invalid dates
if df['date'].isna().any():
    print(f"Warning: {df['date'].isna().sum()} invalid dates found. Dropping rows.")
    df = df.dropna(subset=['date'])

# Aggregate to monthly frequency
df['month_period'] = df['date'].dt.to_period('M')
df = df.groupby(['city', 'month_period']).first().reset_index()
df['date'] = df['month_period'].dt.to_timestamp()
df = df.drop(columns=['month_period'])

# Sort and set index
df = df.sort_values(['city', 'date']).set_index('date')

# Check data sufficiency
cities = df['city'].unique()
for city in cities:
    city_df = df[df['city'] == city]
    if len(city_df) < 12:
        print(f"Warning: {city} has only {len(city_df)} months of data. Skipping (need at least 12).")
        df = df[df['city'] != city]

if df.empty:
    raise ValueError("No cities with sufficient data (at least 12 months).")

# Print data summary
print("\nData Summary:")
print(df.groupby('city').size())
print("\nMissing Values:")
print(df[required_columns[1:]].isna().sum())  # Exclude 'date' since it's the index

# Define targets and exogenous features
targets = ['monsoon_intensity', 'climate_change', 'siltation', 'landslide_risks']
exog_features = ['rainfall_mm', 'temperature_2m', 'runoff', 'total_precipitation', 'high_vegetation_cover',
                 'geopotential_height', 'soil_volume_water_content_level1', 'soil_volume_water_content_level2',
                 'soil_volume_water_content_level3', 'soil_volume_water_content_level4']

# Create future date range (2025-07 to 2050-12)
future_dates = pd.date_range(start='2025-07-01', end='2050-12-01', freq='MS')
n_future = len(future_dates)

# Initialize results
forecasts = []
mae_scores = {}
r2_scores = {}

# Function to forecast exogenous features using SARIMA
def forecast_exog(series, n_future, seasonal_period=12):
    try:
        if len(series) < 12:
            print(f"Insufficient data for exogenous series ({len(series)} months). Using mean.")
            return np.full(n_future, series.mean())
        model = SARIMAX(series, order=(1, 1, 1), seasonal_order=(1, 1, 1, seasonal_period),
                        enforce_stationarity=False, enforce_invertibility=False)
        fitted_model = model.fit(disp=False)
        forecast = fitted_model.forecast(steps=n_future)
        return forecast
    except Exception as e:
        print(f"Error forecasting exogenous series: {e}")
        return np.full(n_future, series.mean())  # Fallback to mean

# Train SARIMAX for each city and target
for city in cities:
    print(f"\nTraining models for city: {city}")
    city_df = df[df['city'] == city].copy()
    
    # Check data sufficiency
    if len(city_df) < 12:
        print(f"Skipping {city}: only {len(city_df)} months of data.")
        continue
    
    # Forecast exogenous features for 2025-07 to 2050-12
    exog_future = pd.DataFrame(index=future_dates)
    for exog in exog_features:
        if exog.startswith('soil_volume_water_content') or exog == 'geopotential_height':
            exog_future[exog] = city_df[exog].iloc[-1]  # Use last known value
        else:
            exog_future[exog] = forecast_exog(city_df[exog], n_future)
    
    for target in targets:
        print(f"  Processing {target}")
        # Prepare endogenous and exogenous data
        endog = city_df[target]
        exog = city_df[exog_features].fillna(city_df[exog_features].mean())
        
        # Split into train (80%) and test (20%)
        train_size = int(len(endog) * 0.8)
        if train_size < 12:
            print(f"  Skipping {target}: insufficient training data ({train_size} months).")
            continue
        train_endog = endog[:train_size]
        train_exog = exog[:train_size]
        test_endog = endog[train_size:]
        test_exog = exog[train_size:]
        
        # Train SARIMAX
        try:
            model = SARIMAX(
                train_endog,
                exog=train_exog,
                order=(1, 1, 1),
                seasonal_order=(1, 1, 1, 12),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            fitted_model = model.fit(disp=False)
            
            # Forecast for test period (for MAE only)
            forecast_test = fitted_model.forecast(steps=len(test_endog), exog=test_exog)
            max_value = 17 if target == 'climate_change' else 16
            forecast_test = np.round(forecast_test).astype(int).clip(0, max_value)
            
            # Calculate MAE
            if len(test_endog) > 0:
                mae = mean_absolute_error(test_endog, forecast_test)
                mae_scores[f"{city}_{target}"] = mae
                print(f"  {target} Test MAE: {mae:.2f}")
            else:
                print(f"  No test data for {target}. Skipping MAE.")

            # Calculate R²
            if len(test_endog) > 0:
                r2 = r2_score(test_endog, forecast_test)
                r2_scores[f"{city}_{target}"] = r2
                print(f"  {target} Test R²: {r2:.2f}")
            else:
                print(f"  No test data for {target}. Skipping R2.")
            
            # Forecast for future period
            forecast_future = fitted_model.forecast(steps=n_future, exog=exog_future)
            forecast_future = np.round(forecast_future).astype(int).clip(0, max_value)
            
            # Store future forecasts only
            forecast_df = pd.DataFrame({
                'date': future_dates,
                'city': city,
                'target': target,
                'forecast': forecast_future
            })
            forecasts.append(forecast_df)
            
        except Exception as e:
            print(f"  Error fitting SARIMAX for {target} in {city}: {e}")
            continue

# Check if forecasts were generated
if not forecasts:
    raise ValueError("No forecasts generated. Check data or model parameters.")

# Combine and pivot forecasts
forecast_df = pd.concat(forecasts, ignore_index=True)
forecast_pivot = forecast_df.pivot_table(
    values='forecast',
    index=['date', 'city'],
    columns='target',
    aggfunc='first'
).reset_index()

# Ensure all target columns are present
for target in targets:
    if target not in forecast_pivot.columns:
        forecast_pivot[target] = np.nan

# Reorder columns
output_columns = ['date', 'city'] + targets
forecast_pivot = forecast_pivot[output_columns]

# Save to CSV
forecast_pivot.to_csv('sarimax_forecasts_2025_2050.csv', index=False)

# Print MAE scores
print("\nTest Mean Absolute Error and R2 Scores:")
for key, value in mae_scores.items():
    print(f"{key}: {value:.2f}")
for key, value in r2_scores.items():
    print(f"{key}: {value:.2f}")
print("\nFuture forecasts (2025-07 to 2050-12) saved to 'sarimax_forecasts_2025_2050.csv'")

### SARIMA Grok

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
csv_file = 'datasets/city_timeseries_synthetic_dataset.csv'  # Update with correct path
try:
    df = pd.read_csv(csv_file, skipinitialspace=True, encoding='utf-8')
except Exception as e:
    print(f"Error loading CSV: {e}")
    raise

# Verify columns
required_columns = ['date', 'city', 'monsoon_intensity', 'climate_change', 'siltation',
                   'landslide_risks', 'rainfall_mm', 'temperature_2m', 'runoff', 'total_precipitation',
                   'high_vegetation_cover', 'geopotential_height', 'soil_volume_water_content_level1',
                   'soil_volume_water_content_level2', 'soil_volume_water_content_level3',
                   'soil_volume_water_content_level4']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}. Available columns: {df.columns.tolist()}")

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce')
if df['date'].isna().any():
    print(f"Warning: {df['date'].isna().sum()} invalid dates found. Dropping rows.")
    df = df.dropna(subset=['date'])

# Aggregate to monthly frequency
df['month_period'] = df['date'].dt.to_period('M')
df = df.groupby(['city', 'month_period']).first().reset_index()
df['date'] = df['month_period'].dt.to_timestamp()
df = df.drop(columns=['month_period'])

# Sort and set index
df = df.sort_values(['city', 'date']).set_index('date')

# Check data sufficiency
cities = df['city'].unique()
for city in cities:
    city_df = df[df['city'] == city]
    if len(city_df) < 24:
        print(f"Warning: {city} has only {len(city_df)} months of data. Skipping.")
        df = df[df['city'] != city]

if df.empty:
    raise ValueError("No cities with sufficient data (at least 24 months).")

# Print data summary
print("\nData Summary:")
print(df.groupby('city').size())
print("\nMissing Values:")
print(df[required_columns[1:]].isna().sum())

# Define targets and exogenous features
targets = ['monsoon_intensity', 'climate_change', 'siltation', 'landslide_risks']
exog_features = ['rainfall_mm', 'temperature_2m', 'runoff', 'total_precipitation', 'high_vegetation_cover',
                 'soil_volume_water_content_level1', 'soil_volume_water_content_level2',
                 'soil_volume_water_content_level3', 'soil_volume_water_content_level4']
dynamic_exog = ['rainfall_mm', 'runoff', 'total_precipitation']
static_exog = [x for x in exog_features if x not in dynamic_exog]

# Create lagged exogenous features
for exog in dynamic_exog:
    df[f'{exog}_lag1'] = df.groupby('city')[exog].shift(1)
exog_features += [f'{exog}_lag1' for exog in dynamic_exog]

# Create future date range (2025-07 to 2050-12)
future_dates = pd.date_range(start='2025-07-01', end='2050-12-01', freq='MS')
n_future = len(future_dates)

# Initialize results
forecasts = []
metrics = []

# Function to check stationarity
def check_stationarity(series):
    result = adfuller(series.dropna())
    return result[1] < 0.05

# Function to detrend climate_change
def detrend_climate_change(series, dates):
    years = (dates - pd.Timestamp('1981-01-01')).days / 365.25
    X = years.reshape(-1, 1)
    model = LinearRegression().fit(X, series)
    trend = model.predict(X)
    detrended = series - trend
    return detrended, model

# Function to forecast exogenous features
def forecast_exog(series, dates, future_dates, seasonal_period=6, is_dynamic=False):
    try:
        if len(series.dropna()) < 12:
            print(f"Insufficient data for exogenous series ({len(series)} months). Using mean.")
            return np.full(len(future_dates), series.mean())
        if is_dynamic:
            if not check_stationarity(series):
                series = series.diff().dropna()
            model = auto_arima(series, seasonal=True, m=seasonal_period, max_p=1, max_q=1,
                               max_P=1, max_Q=1, max_d=1, max_D=0, suppress_warnings=True, stepwise=True, maxiter=10)
            forecast = model.predict(n_periods=len(future_dates))
        else:
            # Use seasonal averages for static variables
            series_df = pd.DataFrame({'value': series}, index=dates)
            series_df['month'] = series_df.index.month
            seasonal_means = series_df.groupby('month')['value'].mean()
            future_months = future_dates.month
            forecast = seasonal_means.reindex(future_months).values
        return np.clip(forecast, 0, None)
    except Exception as e:
        print(f"Error forecasting exogenous series: {e}")
        return np.full(len(future_dates), series.mean())

# Function to process each city and target
def process_city_target(city, target, df, exog_features, future_dates, n_future):
    print(f"Processing {target} for {city}")
    city_df = df[df['city'] == city].copy()
    
    if len(city_df) < 24:
        print(f"Skipping {city}: only {len(city_df)} months of data.")
        return None, None
    
    # Forecast exogenous features
    exog_future = pd.DataFrame(index=future_dates)
    for exog in exog_features:
        if 'lag1' in exog:
            exog_future[exog] = forecast_exog(city_df[exog.replace('_lag1', '')], city_df.index, future_dates,
                                             is_dynamic=exog.replace('_lag1', '') in dynamic_exog)
        else:
            exog_future[exog] = forecast_exog(city_df[exog], city_df.index, future_dates,
                                             is_dynamic=exog in dynamic_exog)
    
    endog = city_df[target].dropna()
    exog = city_df[exog_features].fillna(city_df[exog_features].mean())
    
    # Split into train (80%) and test (20%)
    train_size = int(len(endog) * 0.8)
    if train_size < 12:
        print(f"Skipping {target}: insufficient training data ({train_size} months).")
        return None, None
    
    train_endog = endog[:train_size]
    train_exog = exog[:train_size]
    test_endog = endog[train_size:]
    test_exog = exog[train_size:]
    
    # Define variable-specific ranges for clipping
    max_val = {
        'monsoon_intensity': 1.048475,
        'landslide_risks': 1.584118,
        'climate_change': 1.0,
        'siltation': 1.0
    }[target]
    
    # Preprocess data
    if target == 'climate_change':
        # Detrend climate_change using linear regression
        train_endog_detrended, lr_model = detrend_climate_change(train_endog, train_endog.index)
        test_endog_detrended = test_endog - lr_model.predict((test_endog.index - pd.Timestamp('1981-01-01')).days / 365.25).reshape(-1, 1)
        future_years = (future_dates - pd.Timestamp('1981-01-01')).days / 365.25
        future_trend = np.minimum(lr_model.predict(future_years.reshape(-1, 1)), 1.0)
    else:
        train_endog_detrended = train_endog
        test_endog_detrended = test_endog
        future_trend = 0
    
    # Check stationarity and apply differencing if needed
    if not check_stationarity(train_endog_detrended):
        train_endog_detrended = train_endog_detrended.diff().dropna()
        test_endog_detrended = test_endog_detrended.diff().dropna()
        train_exog = train_exog.loc[train_endog_detrended.index]
        test_exog = test_exog.loc[test_endog_detrended.index]
    
    # Tune SARIMAX parameters
    try:
        if target == 'climate_change':
            model = auto_arima(train_endog_detrended, exogenous=train_exog, seasonal=False,
                               max_p=1, max_q=1, max_d=1, suppress_warnings=True, stepwise=True, maxiter=20)
        else:
            # Try both 6-month and 12-month seasonality for monsoon_intensity
            m = 6 if target == 'monsoon_intensity' else 6
            model = auto_arima(train_endog_detrended, exogenous=train_exog, seasonal=True, m=m,
                               max_p=1, max_q=1, max_P=1, max_Q=1, max_d=1, max_D=0,
                               suppress_warnings=True, stepwise=True, maxiter=20)
        
        # Fit SARIMAX with robust solver
        sarimax_model = SARIMAX(
            train_endog_detrended,
            exog=train_exog,
            order=model.order,
            seasonal_order=model.seasonal_order if target != 'climate_change' else (0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fitted_model = sarimax_model.fit(disp=False, method='nm', maxiter=200)
        
        # Forecast test period
        forecast_test = fitted_model.forecast(steps=len(test_endog), exog=test_exog)
        if target == 'climate_change':
            forecast_test = forecast_test + test_endog_detrended.index.map(
                lambda x: lr_model.predict([[ (x - pd.Timestamp('1981-01-01')).days / 365.25 ]])[0]
            )
        forecast_test = np.clip(forecast_test, 0, max_val)
        
        # Calculate metrics
        metric = {'city': city, 'target': target}
        if len(test_endog) > 0:
            metric['MAE'] = mean_absolute_error(test_endog, forecast_test)
            metric['RMSE'] = np.sqrt(mean_squared_error(test_endog, forecast_test))
            metric['R2'] = r2_score(test_endog, forecast_test)
            print(f"  {target} Test MAE: {metric['MAE']:.4f}, RMSE: {metric['RMSE']:.4f}, R²: {metric['R2']:.4f}")
        
        # Forecast future period
        forecast_future = fitted_model.forecast(steps=n_future, exog=exog_future)
        if target == 'climate_change':
            forecast_future = forecast_future + future_trend
        forecast_future = np.clip(forecast_future, 0, max_val)
        
        # Create forecast DataFrame
        forecast_df = pd.DataFrame({
            'date': future_dates,
            'city': city,
            'target': target,
            'forecast': forecast_future
        })
        
        return forecast_df, metric
    
    except Exception as e:
        print(f"Error fitting SARIMAX for {target} in {city}: {e}")
        return None, None

# Parallel processing for cities and targets
results = Parallel(n_jobs=-1)(
    delayed(process_city_target)(city, target, df, exog_features, future_dates, n_future)
    for city in cities for target in targets
)

# Split results into forecasts and metrics
forecasts = [r[0] for r in results if r[0] is not None]
metrics = [r[1] for r in results if r[1] is not None]

if not forecasts:
    raise ValueError("No forecasts generated. Check data or model parameters.")

# Combine and pivot forecasts
forecast_df = pd.concat(forecasts, ignore_index=True)
forecast_pivot = forecast_df.pivot_table(
    values='forecast',
    index=['date', 'city'],
    columns='target',
    aggfunc='first'
).reset_index()

# Ensure all target columns are present
for target in targets:
    if target not in forecast_pivot.columns:
        forecast_pivot[target] = np.nan

# Reorder columns
output_columns = ['date', 'city', 'monsoon_intensity', 'climate_change', 'siltation', 'landslide_risks']
forecast_pivot = forecast_pivot[output_columns]

# Fill any missing values with mean of the column
forecast_pivot[targets] = forecast_pivot[targets].fillna(forecast_pivot[targets].mean())

# Save to CSV
forecast_pivot.to_csv('sarimax_forecasts_original_ranges_fixed.csv', index=False)

# Print metrics
print("\nTest Metrics:")
metrics_df = pd.DataFrame(metrics)
print(metrics_df.groupby(['city', 'target']).first())
print("\nMonthly forecasts (2025-07 to 2050-12) saved to 'sarimax_forecasts_original_ranges_fixed.csv'")

### Synthetic Flood Prediction Dataset Generator
This notebook creates a realistic (but artificial) dataset for flood risk prediction studies.
Features include climatic, topographic, and human factors, with synthetic flood labels.


In [ ]:
import sdv
print(sdv.__version__)


In [ ]:

# Realistic Synthetic Flood Dataset Generator (SDV 1.24.1+)
# **Using `sdv.single_table.GaussianCopulaSynthesizer` instead of deprecated `TabularPreset`.

import numpy as np
import pandas as pd
from scipy.stats import beta, gamma
import seaborn as sns
import matplotlib.pyplot as plt
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import SingleTableMetadata

# %% [markdown]
"""
## 1. Define Realistic Feature Relationships
"""
# %%
np.random.seed(42)
n_samples = 10_000

# Base distributions (all will be normalized to 0-1 later)
data = pd.DataFrame({
    # Climatic
    "MonsoonIntensity": gamma(a=2, loc=0, scale=0.2).rvs(n_samples),
    "ClimateChange": beta(a=1.5, b=3).rvs(n_samples),
    
    # Topographic/Hydrological
    "TopographyDrainage": beta(a=2, b=2).rvs(n_samples),
    "Watersheds": beta(a=3, b=1).rvs(n_samples),
    "CoastalVulnerability": beta(a=1, b=4).rvs(n_samples),
    
    # Human Factors
    "Urbanization": beta(a=1.2, b=2).rvs(n_samples),
    "Deforestation": beta(a=1.5, b=3).rvs(n_samples),
    "AgriculturalPractices": beta(a=2, b=4).rvs(n_samples),
})

# %% [markdown]
"""
## 2. Generate Correlated Features
"""
# %%
# Step 1: Auto-detect metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data)

# Step 2: Initialize synthesizer
synthesizer = GaussianCopulaSynthesizer(metadata)

# Step 3: Fit and generate data
synthesizer.fit(data)
synthetic_data = synthesizer.sample(num_rows=n_samples)

# Manually enforce key relationships
synthetic_data["WetlandLoss"] = 0.7 * synthetic_data["Urbanization"] + 0.3 * np.random.beta(1, 3, n_samples)
synthetic_data["DrainageSystems"] = 1 - (0.4 * synthetic_data["Urbanization"] + 0.6 * np.random.beta(2, 3, n_samples))

# %% [markdown]
"""
## 3. Add Flood Probability Logic
"""
# %%
weights = {
    "MonsoonIntensity": 0.15,
    "TopographyDrainage": 0.1,
    "DrainageSystems": 0.12,
    "Urbanization": 0.08,
    "WetlandLoss": 0.07,
}

synthetic_data["FloodProbability"] = (
    sum(synthetic_data[feat] * wt for feat, wt in weights.items()) + 
    np.random.normal(0, 0.05, n_samples)
)
synthetic_data["FloodProbability"] = synthetic_data["FloodProbability"].clip(0, 1)

# %% [markdown]
"""
## 4. Export & Visualize
"""
# %%
synthetic_data.to_csv("flood_data_sdv_v1.24.1.csv", index=False)
print("Dataset saved!")

# Plot distributions
plt.figure(figsize=(12, 8))
for i, col in enumerate(synthetic_data.columns[:6]):
    plt.subplot(2, 3, i+1)
    sns.histplot(synthetic_data[col], bins=30, kde=True)
    plt.title(col)
plt.tight_layout()

### ChatGPT synthesis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import SingleTableMetadata

# Settings
np.random.seed(42)
n_samples = 10_000

# Create synthetic "months" to inject seasonality
months = np.random.choice(range(1, 13), size=n_samples)
seasonal_wave = 0.5 + 0.4 * np.sin(2 * np.pi * (months - 2) / 12)  # peaks around April and October

# Base data (independent features with plausible distributions)
data = pd.DataFrame({
    "Month": months,
    "MonsoonIntensity": seasonal_wave + np.random.normal(0, 0.05, n_samples),  # seasonal
    "TopographyDrainage": np.random.beta(2, 2, n_samples),
    "RiverManagement": np.random.beta(2, 2, n_samples),
    "Deforestation": np.random.beta(1.5, 3, n_samples),
    "Urbanization": np.random.beta(1.2, 2, n_samples),
    "ClimateChange": np.random.beta(2, 2.5, n_samples),
    "DamsQuality": np.random.beta(3, 1.5, n_samples),
    "Siltation": np.random.beta(1.5, 2.5, n_samples),
    "AgriculturalPractices": np.random.beta(2, 4, n_samples),
    "Encroachments": np.random.beta(1.8, 3, n_samples),
    "IneffectiveDisasterPreparedness": np.random.beta(2, 3, n_samples),
    "DrainageSystems": 1 - np.random.beta(2, 2, n_samples),  # inverse logic
    "CoastalVulnerability": np.random.beta(1, 4, n_samples),
    "Landslides": seasonal_wave * 0.8 + np.random.beta(2, 5, n_samples),
    "Watersheds": np.random.beta(3, 1, n_samples),
    "DeterioratingInfrastructure": np.random.beta(2, 3, n_samples),
    "PopulationScore": np.random.beta(2.5, 2, n_samples),
    "InadequatePlanning": np.random.beta(2, 3, n_samples),
    "PoliticalFactors": np.random.beta(2, 2, n_samples),
})

# Derived fields (inject correlations)
data["WetlandLoss"] = (
    0.6 * data["Urbanization"] +
    0.3 * data["Encroachments"] +
    0.1 * np.random.beta(1, 3, n_samples)
)

data["DrainageSystems"] = (
    1 - (0.5 * data["Urbanization"] + 0.3 * data["Siltation"] + 0.2 * np.random.beta(2, 3, n_samples))
).clip(0, 1)

data["RiverManagement"] = (
    data["RiverManagement"] * (1 - 0.3 * data["PoliticalFactors"])
).clip(0, 1)

# Flood Probability Calculation (weighted sum of key risk factors)
weights = {
    "MonsoonIntensity": 0.15,
    "TopographyDrainage": 0.1,
    "DrainageSystems": 0.1,
    "Urbanization": 0.07,
    "WetlandLoss": 0.07,
    "ClimateChange": 0.06,
    "Landslides": 0.05,
    "RiverManagement": -0.05,  # good management lowers risk
    "DamsQuality": -0.04,
    "Siltation": 0.04,
    "PopulationScore": 0.04,
    "InadequatePlanning": 0.04,
    "Deforestation": 0.03,
    "IneffectiveDisasterPreparedness": 0.03,
    "Encroachments": 0.02,
    "DeterioratingInfrastructure": 0.02
}

data["FloodProbability"] = sum(data[feat] * wt for feat, wt in weights.items())
data["FloodProbability"] += np.random.normal(0, 0.05, n_samples)  # noise
data["FloodProbability"] = data["FloodProbability"].clip(0, 1)

# Remove Month (optional)
data.drop(columns=["Month"], inplace=True)

# Ensure metadata for SDV
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data.drop(columns=["FloodProbability"]))  # exclude target for SDV modeling

# Train synthesizer
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(data.drop(columns=["FloodProbability"]))

# Sample new synthetic data (optional future use)
# synthetic_data = synthesizer.sample(num_rows=5000)

# Export final dataset
data.to_csv("synthetic_flood_dataset_uganda_seasonal.csv", index=False)
print("✅ Dataset exported: synthetic_flood_dataset_uganda_seasonal.csv")

# Quick distribution plot
plt.figure(figsize=(14, 8))
for i, col in enumerate(data.columns[:9]):
    plt.subplot(3, 3, i + 1)
    sns.histplot(data[col], bins=30, kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()


### Gemini 2.5 Pro Sythensis

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Constants ---
N_SAMPLES = 15_000  # Number of data points (e.g., daily readings over ~41 years)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


# Create a date range
dates = pd.to_datetime(pd.date_range(start="1985-01-01", periods=N_SAMPLES, freq='D'))
time_factor = np.linspace(0, 1, N_SAMPLES) # Represents the progression of time for trends

# Bimodal seasonal pattern for Uganda (1 = peak rainy season, 0 = dry season)
month = dates.month
# Coefficients are set to create peaks in April-May and Oct-Nov
seasonal_rainfall_factor = (
    0.5 * (1 + np.cos(2 * np.pi * (month - 4.5) / 12))**4 + # 1st peak (around April/May)
    0.9 * (1 + np.cos(2 * np.pi * (month - 10.5) / 12))**6 # 2nd, stronger peak (around Oct/Nov)
)
seasonal_rainfall_factor = (seasonal_rainfall_factor - seasonal_rainfall_factor.min()) / \
                           (seasonal_rainfall_factor.max() - seasonal_rainfall_factor.min())



df = pd.DataFrame(index=dates)

# --- Foundational Factors ---
df['ClimateChange'] = time_factor * 0.7 + np.random.normal(0, 0.05, N_SAMPLES)
df['PoliticalFactors'] = 0.5 * (1 + np.sin(2 * np.pi * time_factor * 2)) + np.random.normal(0, 0.1, N_SAMPLES)
df['PopulationScore'] = time_factor * 0.8 + beta(a=2, b=5).rvs(N_SAMPLES) * 0.2

# --- Human & Infrastructure Factors ---
df['Urbanization'] = df['PopulationScore'] * 0.6 + time_factor * 0.2 + np.random.normal(0, 0.05, N_SAMPLES)
df['Encroachments'] = 0.7 * df['Urbanization'] + 0.3 * df['PopulationScore'] + np.random.normal(0, 0.05, N_SAMPLES)
df['Deforestation'] = 0.5 * df['Urbanization'] + 0.3 * df['PopulationScore'] + np.random.normal(0, 0.1, N_SAMPLES)
df['AgriculturalPractices'] = 0.4 * df['PopulationScore'] + np.random.beta(2, 3, N_SAMPLES) * 0.6
df['WetlandLoss'] = 0.6 * df['Urbanization'] + 0.3 * df['AgriculturalPractices'] + np.random.normal(0, 0.05, N_SAMPLES)
df['InadequatePlanning'] = 0.5 * df['PoliticalFactors'] + 0.4 * df['Urbanization'] + np.random.normal(0, 0.1, N_SAMPLES)
df['DeterioratingInfrastructure'] = 0.5 * time_factor + 0.3 * df['InadequatePlanning'] + np.random.normal(0, 0.1, N_SAMPLES)
df['DamsQuality'] = 1 - (0.7 * df['DeterioratingInfrastructure'] + np.random.normal(0, 0.1, N_SAMPLES))
df['DrainageSystems'] = 1 - (0.5 * df['Urbanization'] + 0.4 * df['DeterioratingInfrastructure'] + np.random.normal(0, 0.08, N_SAMPLES))

# --- Environmental & Hydrological Factors ---
# Re-interpreting "Coastal" as "Lakeshore" for landlocked Uganda
df['CoastalVulnerability'] = 0.4 * df['ClimateChange'] + 0.4 * df['WetlandLoss'] + np.random.normal(0, 0.1, N_SAMPLES)
df['Siltation'] = 0.6 * df['Deforestation'] + 0.4 * df['AgriculturalPractices'] + np.random.normal(0, 0.05, N_SAMPLES)
df['TopographyDrainage'] = beta(a=2.5, b=2.5).rvs(N_SAMPLES) # Static geographical feature
df['Watersheds'] = 1 - (0.5 * df['Siltation'] + 0.3 * df['Deforestation'] + 0.2 * df['Urbanization'])
df['RiverManagement'] = 1 - (0.6 * df['InadequatePlanning'] + 0.3 * df['Siltation'] + np.random.normal(0, 0.1, N_SAMPLES))

# --- Primary Driver & Compounding Factors ---
df['MonsoonIntensity'] = (0.7 * seasonal_rainfall_factor + 0.3 * df['ClimateChange'] + np.random.normal(0, 0.08, N_SAMPLES))
df['Landslides'] = 0.5 * df['MonsoonIntensity'] + 0.3 * df['Deforestation'] + 0.2 * (1 - df['TopographyDrainage'])

# --- Disaster Preparedness ---
df['IneffectiveDisasterPreparedness'] = 0.5 * df['InadequatePlanning'] + 0.4 * df['PoliticalFactors'] + np.random.normal(0, 0.1, N_SAMPLES)

# Clip all values to be strictly between 0 and 1
for col in df.columns:
    df[col] = np.clip(df[col], 0.01, 1.0)
    
 
# `FloodProbability` is calculated as a weighted sum of the most critical contributing factors. This simulates how different elements combine to create a flood event.
# %%
weights = {
    # Primary Drivers (High Impact)
    "MonsoonIntensity": 0.25,
    "DrainageSystems": -0.15,  # Negative weight: good systems REDUCE probability
    "Siltation": 0.10,
    "TopographyDrainage": -0.10, # Negative weight: good drainage REDUCE probability
    
    # Secondary Drivers (Medium Impact)
    "RiverManagement": -0.08, # Negative weight: good management REDUCES probability
    "Deforestation": 0.07,
    "Urbanization": 0.07,
    "Landslides": 0.06,
    "IneffectiveDisasterPreparedness": 0.06,
    "WetlandLoss": 0.05,
    "CoastalVulnerability": 0.05, # Lakeshore vulnerability
}

# Calculate raw score
flood_score = (
    weights["MonsoonIntensity"] * df["MonsoonIntensity"] +
    weights["DrainageSystems"] * (1 - df["DrainageSystems"]) + # Invert for risk
    weights["Siltation"] * df["Siltation"] +
    weights["TopographyDrainage"] * (1 - df["TopographyDrainage"]) + # Invert for risk
    weights["RiverManagement"] * (1 - df["RiverManagement"]) + # Invert for risk
    weights["Deforestation"] * df["Deforestation"] +
    weights["Urbanization"] * df["Urbanization"] +
    weights["Landslides"] * df["Landslides"] +
    weights["IneffectiveDisasterPreparedness"] * df["IneffectiveDisasterPreparedness"] +
    weights["WetlandLoss"] * df["WetlandLoss"] +
    weights["CoastalVulnerability"] * df["CoastalVulnerability"]
)

# Normalize to a 0-1 probability and add noise
df['FloodProbability'] = (flood_score - flood_score.min()) / (flood_score.max() - flood_score.min())
df['FloodProbability'] = (df['FloodProbability']**1.5 + np.random.normal(0, 0.03, N_SAMPLES))
df['FloodProbability'] = np.clip(df['FloodProbability'], 0, 1)


# Reorder columns to match the user's request
final_columns = [
    "MonsoonIntensity", "TopographyDrainage", "RiverManagement", "Deforestation", 
    "Urbanization", "ClimateChange", "DamsQuality", "Siltation", "AgriculturalPractices", 
    "Encroachments", "IneffectiveDisasterPreparedness", "DrainageSystems", 
    "CoastalVulnerability", "Landslides", "Watersheds", "DeterioratingInfrastructure", 
    "PopulationScore", "WetlandLoss", "InadequatePlanning", "PoliticalFactors", 
    "FloodProbability"
]
final_df = df[final_columns]

# --- Export ---
final_df.to_csv("synthetic_flood_risk_dataset_gemini.csv")
print("✅ Synthetic dataset successfully generated and saved!")
print(final_df.head())

# --- Visualize ---
# 1. Seasonal Pattern Verification
plt.figure(figsize=(14, 6))
plt.plot(final_df.index, final_df['MonsoonIntensity'], label='Generated Monsoon Intensity', alpha=0.7)
plt.plot(df.index, seasonal_rainfall_factor, label='Base Seasonal Factor', color='red', linestyle='--', lw=2)
plt.title('Verification of Seasonal Rainfall Pattern (Uganda-like)')
plt.xlabel('Date')
plt.ylabel('Normalized Intensity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 2. Correlation Heatmap
plt.figure(figsize=(18, 15))
sns.heatmap(final_df.corr(), cmap='coolwarm', annot=False) # annot=True is too crowded
plt.title('Correlation Matrix of Synthetic Flood Features')
plt.show()

### ChatGPT synthesis of scoring dataset.

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Seed for reproducibility
np.random.seed(42)

# Define cities with coordinates and fixed attributes
cities = {
    "Kampala": {
        "lat": 0.3476,
        "lon": 32.5825,
        "grid_lat": -1.0,
        "grid_lon": 30.0,
        "land_sea_mask": 0.927990478079746,
        "lake_cover": 0.0720095219202539,
        "geopotential_height": 11573.29113153154
    },
    "Kabale": {
        "lat": -1.2483,
        "lon": 29.9856,
        "grid_lat": -1.0,
        "grid_lon": 30.0,
        "land_sea_mask": 1.0,
        "lake_cover": 0.0,
        "geopotential_height": 17455.652201526806
    },
    "Kasese": {
        "lat": 0.1833,
        "lon": 30.0833,
        "grid_lat": -1.0,
        "grid_lon": 30.0,
        "land_sea_mask": 1.0,
        "lake_cover": 0.02,
        "geopotential_height": 10529.614022288868
    }
}

# Generate monthly dates
start_date = datetime(1981, 1, 1)
end_date = datetime(2025, 6, 1)
dates = []
while start_date <= end_date:
    dates.append(start_date)
    start_date += relativedelta(months=1)

# Generate synthetic dataset
rows = []

for city, attr in cities.items():
    for date in dates:
        month = date.month

        # Seasonal waveform to simulate Uganda's bimodal rainy pattern
        seasonal_wave = 0.5 + 0.4 * np.sin(2 * np.pi * (month - 2) / 12)

        # Climate/environmental variables
        total_precipitation = max(0, np.random.gamma(2, seasonal_wave * 2))
        runoff = total_precipitation * np.random.uniform(0.1, 0.3)
        evaporation = np.random.normal(0.004, 0.001)
        dewpoint_temperature_2m = np.random.normal(285 + seasonal_wave * 5, 1.5)

        surface_pressure = np.random.normal(82900, 100)
        temperature_2m = np.random.normal(290 + seasonal_wave * 5, 1.2)

        u_wind = np.random.normal(0, 0.5)
        v_wind = np.random.normal(0, 0.5)

        # Soil & skin attributes
        soil_temps = [temperature_2m + np.random.normal(-1.5 + i * 0.2, 0.3) for i in range(4)]
        soil_volume_water_content = [np.random.beta(2, 5) for _ in range(4)]
        skin_reservoir_content = np.random.beta(2, 4) * 0.001
        skin_temperature = temperature_2m + np.random.normal(1.5, 0.5)

        # Vegetation & land
        high_vegetation_cover = np.random.beta(4, 2)
        high_vegetation_type = 19.0
        low_vegetation_cover = np.random.beta(2, 5)
        low_vegetation_type = 1.00003
        soil_type = 2.00003

        # Derived targets (for core model)
        monsoon_intensity = total_precipitation + runoff - evaporation
        climate_change = 5 + 0.02 * (date.year - 1981) + np.random.normal(0, 0.2)
        siltation = runoff + np.mean(soil_volume_water_content) + np.random.normal(0, 0.1)
        landslide_risks = (
            (1 if city == "Kabale" else 0.7 if city == "Kasese" else 0.4)
            * total_precipitation * 0.5
            + high_vegetation_cover * -0.3
            + np.mean(soil_volume_water_content) * 0.5
        )

        # Row
        row = {
            "city": city,
            "date": date.strftime("%Y-%m-%d"),
            "geopotential_height": attr["geopotential_height"],
            "high_vegetation_cover": high_vegetation_cover,
            "high_vegetation_type": high_vegetation_type,
            "lake_cover": attr["lake_cover"],
            "land_sea_mask": attr["land_sea_mask"],
            "low_vegetation_cover": low_vegetation_cover,
            "low_vegetation_type": low_vegetation_type,
            "soil_type": soil_type,
            "target_latitude": attr["lat"],
            "target_longitude": attr["lon"],
            "grid_latitude": attr["grid_lat"],
            "grid_longitude": attr["grid_lon"],
            "total_precipitation": total_precipitation,
            "runoff": runoff,
            "evaporation": evaporation,
            "dewpoint_temperature_2m": dewpoint_temperature_2m,
            "experiment_version": 1.0,
            "skin_reservoir_content": skin_reservoir_content,
            "skin_temperature": skin_temperature,
            "soil_temperature_level1": soil_temps[0],
            "soil_temperature_level2": soil_temps[1],
            "soil_temperature_level3": soil_temps[2],
            "soil_temperature_level4": soil_temps[3],
            "soil_volume_water_content_level1": soil_volume_water_content[0],
            "soil_volume_water_content_level2": soil_volume_water_content[1],
            "soil_volume_water_content_level3": soil_volume_water_content[2],
            "soil_volume_water_content_level4": soil_volume_water_content[3],
            "surface_pressure": surface_pressure,
            "temperature_2m": temperature_2m,
            "u_component_wind_10m": u_wind,
            "v_component_wind_10m": v_wind,
            "rainfall_mm": total_precipitation * 1000,
            "rainfall_monthly_anomaly": np.random.normal(0, 25),
            "rainfall_3month_avg": total_precipitation * 1000 + np.random.normal(0, 15),
            "rainfall_6month_avg": total_precipitation * 1000 + np.random.normal(0, 30),
            "monsoon_intensity": monsoon_intensity,
            "climate_change": climate_change,
            "siltation": siltation,
            "landslide_risks": landslide_risks
        }

        rows.append(row)

# Final dataset
df = pd.DataFrame(rows)

# Save to CSV
df.to_csv("city_timeseries_synthetic_dataset.csv", index=False)
print("✅ Dataset saved: city_timeseries_synthetic_dataset.csv")


### ChatGPT Synthentic Forecasted Dataset.

In [38]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

np.random.seed(42)

# Cities and their attributes
cities = {
    "Kampala": {
        "lat": 0.3476,
        "lon": 32.5825,
        "grid_lat": -1.0,
        "grid_lon": 30.0,
        "land_sea_mask": 0.927990478079746,
        "lake_cover": 0.0720095219202539,
        "geopotential_height": 11573.29113153154
    },
    "Kabale": {
        "lat": -1.2483,
        "lon": 29.9856,
        "grid_lat": -1.0,
        "grid_lon": 30.0,
        "land_sea_mask": 1.0,
        "lake_cover": 0.0,
        "geopotential_height": 17455.652201526806
    },
    "Kasese": {
        "lat": 0.1833,
        "lon": 30.0833,
        "grid_lat": -1.0,
        "grid_lon": 30.0,
        "land_sea_mask": 1.0,
        "lake_cover": 0.02,
        "geopotential_height": 10529.614022288868
    }
}

# Define future date range from July 2025 to December 2050
start_date = datetime(2025, 7, 1)
end_date = datetime(2050, 12, 1)
dates = []
while start_date <= end_date:
    dates.append(start_date)
    start_date += relativedelta(months=1)

# Target variable scaling boundaries
scaler_targets = {
    "monsoon_intensity": (0.0, 1.0),
    "climate_change": (0.0, 1.0),
    "siltation": (0.0, 1.0 ),
    "landslide_risks": (0.0, 1.0),
}

def rescale_to_target(value, original_min, original_max, variable_name):
    """
    Rescales a single value from its original range to a specific target range
    defined in scaler_targets, using min-max normalization.

    Args:
        value: The raw value to be rescaled.
        original_min: The known minimum of the original raw data for this variable.
        original_max: The known maximum of the original raw data for this variable.
        variable_name: The string key for the variable (e.g., "monsoon_intensity")
                       to look up its target range in scaler_targets.

    Returns:
        The rescaled value within the specified target range.
        Returns original_min if original_max == original_min to avoid division by zero.
        Returns None if variable_name is not found in scaler_targets.
    """
    if variable_name not in scaler_targets:
        print(f"Error: '{variable_name}' not found in scaler_targets.")
        return None

    target_min, target_max = scaler_targets[variable_name]

    # Handle the case where original_min and original_max are the same
    if original_max == original_min:
        return target_min

    # Min-max scaling formula
    scaled_value = target_min + (value - original_min) * \
                   (target_max - target_min) / (original_max - original_min)
    return scaled_value

# Utility function for scaling
def scale_value(value, min_val, max_val):
    return np.clip(value, min_val, max_val)

# Generate dataset
rows = []

for city, attr in cities.items():
    for date in dates:
        month = date.month
        seasonal_wave = 0.5 + 0.4 * np.sin(2 * np.pi * (month - 2) / 12)

        total_precipitation = max(0, np.random.gamma(2, seasonal_wave * 2))
        runoff = total_precipitation * np.random.uniform(0.1, 0.3)
        evaporation = np.random.normal(0.004, 0.001)
        dewpoint_temperature_2m = np.random.normal(285 + seasonal_wave * 5, 1.5)

        surface_pressure = np.random.normal(82900, 100)
        temperature_2m = np.random.normal(290 + seasonal_wave * 5, 1.2)

        u_wind = np.random.normal(0, 0.5)
        v_wind = np.random.normal(0, 0.5)

        soil_temps = [temperature_2m + np.random.normal(-1.5 + i * 0.2, 0.3) for i in range(4)]
        soil_volume_water_content = [np.random.beta(2, 5) for _ in range(4)]
        skin_reservoir_content = np.random.beta(2, 4) * 0.001
        skin_temperature = temperature_2m + np.random.normal(1.5, 0.5)

        high_vegetation_cover = np.random.beta(4, 2)
        high_vegetation_type = 19.0
        low_vegetation_cover = np.random.beta(2, 5)
        low_vegetation_type = 1.00003
        soil_type = 2.00003

        # Unscaled derived variables
        raw_monsoon_intensity = total_precipitation + runoff - evaporation
        raw_climate_change = 5 + 0.02 * (date.year - 1981) + np.random.normal(0, 0.2)
        raw_siltation = runoff + np.mean(soil_volume_water_content) + np.random.normal(0, 0.1)
        raw_landslide_risks = (
            (0.8 if city == "Kabale" else 0.5 if city == "Kasese" else 0.2)
            * total_precipitation * 0.5
            + high_vegetation_cover * -0.3
            + np.mean(soil_volume_water_content) * 0.5
        )

        # Scale derived targets
        # monsoon_intensity = scale_value(raw_monsoon_intensity, *scaler_targets["monsoon_intensity"])
        # climate_change = scale_value(raw_climate_change / 6, *scaler_targets["climate_change"])  # normalize
        # siltation = scale_value(raw_siltation / 3, *scaler_targets["siltation"])  # normalize
        # landslide_risks = scale_value(raw_landslide_risks / 2.5, *scaler_targets["landslide_risks"])  # normalize

        monsoon_intensity = rescale_to_target(raw_monsoon_intensity, 0.023189 , 17.654824, "monsoon_intensity")
        climate_change = rescale_to_target(raw_climate_change, 5.360623, 6.879625, "climate_change")
        siltation = rescale_to_target(raw_siltation, -0.060039, 3.699172, "siltation")
        landslide_risks = rescale_to_target(raw_landslide_risks, -0.165820, 4.750064, "landslide_risks")

        row = {
            "city": city,
            "date": date.strftime("%Y-%m-%d"),
            "geopotential_height": attr["geopotential_height"],
            "high_vegetation_cover": high_vegetation_cover,
            "high_vegetation_type": high_vegetation_type,
            "lake_cover": attr["lake_cover"],
            "land_sea_mask": attr["land_sea_mask"],
            "low_vegetation_cover": low_vegetation_cover,
            "low_vegetation_type": low_vegetation_type,
            "soil_type": soil_type,
            "target_latitude": attr["lat"],
            "target_longitude": attr["lon"],
            "grid_latitude": attr["grid_lat"],
            "grid_longitude": attr["grid_lon"],
            "total_precipitation": total_precipitation,
            "runoff": runoff,
            "evaporation": evaporation,
            "dewpoint_temperature_2m": dewpoint_temperature_2m,
            "experiment_version": 2.0,
            "skin_reservoir_content": skin_reservoir_content,
            "skin_temperature": skin_temperature,
            "soil_temperature_level1": soil_temps[0],
            "soil_temperature_level2": soil_temps[1],
            "soil_temperature_level3": soil_temps[2],
            "soil_temperature_level4": soil_temps[3],
            "soil_volume_water_content_level1": soil_volume_water_content[0],
            "soil_volume_water_content_level2": soil_volume_water_content[1],
            "soil_volume_water_content_level3": soil_volume_water_content[2],
            "soil_volume_water_content_level4": soil_volume_water_content[3],
            "surface_pressure": surface_pressure,
            "temperature_2m": temperature_2m,
            "u_component_wind_10m": u_wind,
            "v_component_wind_10m": v_wind,
            "rainfall_mm": total_precipitation * 1000,
            "rainfall_monthly_anomaly": np.random.normal(0, 25),
            "rainfall_3month_avg": total_precipitation * 1000 + np.random.normal(0, 15),
            "rainfall_6month_avg": total_precipitation * 1000 + np.random.normal(0, 30),
            "monsoon_intensity": monsoon_intensity,
            "climate_change": climate_change,
            "siltation": siltation,
            "landslide_risks": landslide_risks
        }

        rows.append(row)

# Save DataFrame
df_future = pd.DataFrame(rows)
df_future.to_csv("city_timeseries_future_2025_2050_rescaled.csv", index=False)
print("✅ Future dataset saved: city_timeseries_future_2025_2050.csv")


✅ Future dataset saved: city_timeseries_future_2025_2050.csv


### Static Rescaler

In [37]:
import numpy as np
import pandas as pd

# Target variable scaling boundaries
scaler_targets = {
    "TopographyDrainage": (0.0, 1.0),
    "RiverManagement": (0.0, 1.0),
    "Deforestation": (0.0, 1.0),
    "Urbanization": (0.0, 1.0),
    "DamsQuality": (0.0, 1.0),
    "AgriculturalPractices": (0.0, 1.0),
    "Encroachments": (0.0, 1.0),
    "IneffectiveDisasterPreparedness": (0.0, 1.0),
    "DrainageSystems": (0.0, 1.0),
    "CoastalVulnerability": (0.0, 1.0),
    "Watersheds": (0.0, 1.0),
    "DeterioratingInfrastructure": (0.0, 1.0),
    "PopulationScore": (0.0, 1.0),
    "WetlandLoss": (0.0, 1.0),
    "InadequatePlanning": (0.0, 1.0),
    "PoliticalFactors": (0.0, 1.0),
}

def rescale(value, original_min, original_max, variable_name):
    """
    Rescales a single value from its original range to a specific target range
    defined in scaler_targets, using min-max normalization.

    Args:
        value: The raw value to be rescaled.
        original_min: The known minimum of the original raw data for this variable.
        original_max: The known maximum of the original raw data for this variable.
        variable_name: The string key for the variable (e.g., "monsoon_intensity")
                       to look up its target range in scaler_targets.

    Returns:
        The rescaled value within the specified target range.
        Returns original_min if original_max == original_min to avoid division by zero.
        Returns None if variable_name is not found in scaler_targets.
    """
    if variable_name not in scaler_targets:
        print(f"Error: '{variable_name}' not found in scaler_targets.")
        return None

    target_min, target_max = scaler_targets[variable_name]

    # Handle the case where original_min and original_max are the same
    if original_max == original_min:
        return target_min

    # Min-max scaling formula
    scaled_value = target_min + (value - original_min) * \
                   (target_max - target_min) / (original_max - original_min)
    return scaled_value

df = pd.read_csv('datasets/static/static_features_uganda_cities_.csv')

# Rescale each feature using the defined scaler_targets
for col in df.columns:
    if col in scaler_targets:
        original_min = df[col].min()
        original_max = df[col].max()
        df[col] = df[col].apply(lambda x: rescale(x, 0, 16, col))

df.describe()
# # Save the rescaled DataFrame
df.to_csv('datasets/static/static_features_uganda_cities_rescaled.csv', index=False)